In [ ]:
#Part -4  LLM-Powered Feature:  Model Prediction Explanation

# LOAD THE API KEY
from google.colab import userdata

api_key = userdata.get("LLM_API_KEY")

if api_key:
    print("✅ API key loaded successfully!")
else:
    print("❌ API key not found.")

✅ API key loaded successfully!


In [ ]:
#Load the cleaned dataset
import pandas as pd

df = pd.read_csv("stroke_cleaned_data.csv")

print(df.head())

   gender   age  hypertension  heart_disease ever_married      work_type  \
0    Male  67.0             0              1          Yes        Private   
1  Female  61.0             0              0          Yes  Self-employed   
2    Male  80.0             0              1          Yes        Private   
3  Female  49.0             0              0          Yes        Private   
4  Female  79.0             1              0          Yes  Self-employed   

  Residence_type  avg_glucose_level   bmi   smoking_status  stroke  
0          Urban             228.69  36.6  formerly smoked       1  
1          Rural             202.21  28.1     never smoked       1  
2          Rural             105.92  32.5     never smoked       1  
3          Urban             171.23  34.4           smokes       1  
4          Rural             174.12  24.0     never smoked       1  


In [ ]:
from google.colab import userdata
import os

os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")

api_key = os.environ["LLM_API_KEY"]

print("✅ API key loaded successfully!")

✅ API key loaded successfully!


In [ ]:
# ===============================
# Task 1 - Set up LLM API Connection
# ===============================

import os
import requests
from google.colab import userdata

# --------------------------------
# Load API Key from Colab Secrets
# --------------------------------
os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
api_key = os.environ["LLM_API_KEY"]

# --------------------------------
# OpenRouter API URL
# --------------------------------
url = "https://openrouter.ai/api/v1/chat/completions"

# --------------------------------
# Reusable LLM Function
# --------------------------------
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    # JSON Payload
    payload = {
        "model": "openai/gpt-4o-mini",
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    # Request Headers
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # Send POST Request
    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    # Check Response Status
    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    # Return LLM Response
    return response.json()["choices"][0]["message"]["content"]

# --------------------------------
# Test the Function
# --------------------------------

system_prompt = "You are a helpful assistant."

user_prompt = "Reply with only the word: hello"

result = call_llm(system_prompt, user_prompt)

print(result)

hello


In [ ]:
# ============================================
# Task 2 - Prompt Design (Track C)
# ============================================

system_prompt =""" You are a medical prediction explanation assistant.

You will receive:
- Patient feature values
- A predicted class
- A predicted probability

Your task is to explain the prediction and return ONLY valid JSON.

The JSON must contain exactly the following fields:

{
  "prediction_label": "string",
  "confidence_level": "low | medium | high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}

Do not include markdown.
Do not include code fences.
Do not include any extra explanation outside the JSON.
"""
# User Prompt Template (sample values for testing)
user_prompt = """
Patient Features

Age: 56
Hypertension: Yes
Heart Disease: Yes
BMI: 36.5

Predicted Class:
Stroke

Predicted Probability:
0.89

Explain this prediction and return only valid JSON.
"""

# -------------------------------
# Temperature = 0
# -------------------------------
print("===== Temperature = 0 =====")

response_temp0 = call_llm(
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    temperature=0.0,
    max_tokens=512
)

print(response_temp0)


# -------------------------------
# Temperature = 0.7
# -------------------------------
print("\n===== Temperature = 0.7 =====")

response_temp07 = call_llm(
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    temperature=0.7,
    max_tokens=512
)

print(response_temp07)

===== Temperature = 0 =====
{
  "prediction_label": "Stroke",
  "confidence_level": "high",
  "top_reason": "The patient has a high BMI, which is a significant risk factor for stroke.",
  "second_reason": "The presence of both hypertension and heart disease further increases the risk of stroke.",
  "next_step": "Recommend immediate medical evaluation and management of risk factors."
}

===== Temperature = 0.7 =====
{
  "prediction_label": "Stroke",
  "confidence_level": "high",
  "top_reason": "The patient has a high BMI of 36.5, which is classified as obese and is a significant risk factor for stroke.",
  "second_reason": "The presence of hypertension and heart disease further increases the likelihood of stroke.",
  "next_step": "Immediate medical evaluation and intervention are recommended to manage risk factors and prevent stroke."
}


In [3]:
import joblib

model = joblib.load("best_model.pkl")
print(type(model))

<class 'sklearn.pipeline.Pipeline'>


In [4]:
# upload the best model file
from google.colab import files

files.upload()

Saving best_model.pkl to best_model (1).pkl


{'best_model (1).pkl': b'\x80\x04\x95\xc1\x01\x00\x00\x00\x00\x00\x00\x8c\x10sklearn.pipeline\x94\x8c\x08Pipeline\x94\x93\x94)\x81\x94}\x94(\x8c\x05steps\x94]\x94(\x8c\rsimpleimputer\x94\x8c\x14sklearn.impute._base\x94\x8c\rSimpleImputer\x94\x93\x94)\x81\x94}\x94(\x8c\x0emissing_values\x94G\x7f\xf8\x00\x00\x00\x00\x00\x00\x8c\radd_indicator\x94\x89\x8c\x13keep_empty_features\x94\x89\x8c\x08strategy\x94\x8c\x06median\x94\x8c\nfill_value\x94N\x8c\x04copy\x94\x88\x8c\x11feature_names_in_\x94\x8c\x13joblib.numpy_pickle\x94\x8c\x11NumpyArrayWrapper\x94\x93\x94)\x81\x94}\x94(\x8c\x08subclass\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94\x8c\x05shape\x94K\x0f\x85\x94\x8c\x05order\x94\x8c\x01C\x94\x8c\x05dtype\x94h\x1b\x8c\x05dtype\x94\x93\x94\x8c\x02O8\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01|\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK?t\x94b\x8c\nallow_mmap\x94\x89\x8c\x1bnumpy_array_alignment_bytes\x94K\x10ub\x80\x05\x95\xaf\x01\x00\x00\x00\x00\x00\x00\x8c\x16numpy._core.multiarray\x94\x8c\x

In [2]:
import joblib

model = joblib.load("best_model.pkl")
print(type(model))

<class 'sklearn.pipeline.Pipeline'>


In [ ]:
#model loading
import joblib

best_model = joblib.load("best_model.pkl")

print("✅ Model loaded successfully!")
print(type(best_model))
print(best_model)

✅ Model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=5, min_samples_leaf=5,
                                        n_estimators=200, random_state=42))])


In [ ]:
#Task - 3
#Structured output handling

# Import Libraries & Load Model


import joblib
import json
import pandas as pd
from jsonschema import validate, ValidationError

# Load the saved best model
best_model = joblib.load("best_model.pkl")

print("Best model loaded successfully!")




Best model loaded successfully!


In [ ]:
#Creating Three Hand-Crafted Patient Records

patient1 = {
    "age": 65,
    "hypertension": 1,
    "heart_disease": 1,
    "bmi": 32.5,
    "gender_Male": 1,
    "gender_Other": 0,
    "ever_married_Yes": 1,
    "work_type_Never_worked": 0,
    "work_type_Private": 1,
    "work_type_Self-employed": 0,
    "work_type_children": 0,
    "Residence_type_Urban": 1,
    "smoking_status_formerly smoked": 1,
    "smoking_status_never smoked": 0,
    "smoking_status_smokes": 0
}

patient2 = {
    "age": 30,
    "hypertension": 0,
    "heart_disease": 0,
    "bmi": 23.5,
    "gender_Male": 0,
    "gender_Other": 0,
    "ever_married_Yes": 0,
    "work_type_Never_worked": 0,
    "work_type_Private": 1,
    "work_type_Self-employed": 0,
    "work_type_children": 0,
    "Residence_type_Urban": 0,
    "smoking_status_formerly smoked": 0,
    "smoking_status_never smoked": 1,
    "smoking_status_smokes": 0
}

patient3 = {
    "age": 75,
    "hypertension": 1,
    "heart_disease": 1,
    "bmi": 36.8,
    "gender_Male": 0,
    "gender_Other": 0,
    "ever_married_Yes": 1,
    "work_type_Never_worked": 0,
    "work_type_Private": 0,
    "work_type_Self-employed": 1,
    "work_type_children": 0,
    "Residence_type_Urban": 1,
    "smoking_status_formerly smoked": 0,
    "smoking_status_never smoked": 0,
    "smoking_status_smokes": 1
}

patients = [patient1, patient2, patient3]

print("Three patient records created.")

Three patient records created.


In [ ]:
#Encode the Patient Records

def encode_record(features):
    return pd.DataFrame([features])

print("Encoding function created.")

Encoding function created.


In [ ]:
# Predict Class & Probability


prediction_results = []

for patient in patients:

    encoded = encode_record(patient)

    predicted_class = int(best_model.predict(encoded)[0])

    predicted_probability = float(
        best_model.predict_proba(encoded)[0][1]
    )

    prediction_results.append({
        "features": patient,
        "predicted_class": predicted_class,
        "predicted_probability": predicted_probability
    })

print("Predictions completed.")

Predictions completed.


In [ ]:
# JSON Schema

schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

print("Schema created.")

Schema created.


In [ ]:
# LLM Explanation & Validation

for result in prediction_results:

    prediction = "Stroke" if result["predicted_class"] == 1 else "No Stroke"

    system_prompt = """
You are a medical prediction explanation assistant.

You will receive:
- Patient feature values
- Predicted class
- Predicted probability

Return ONLY valid JSON.

The JSON must contain exactly these fields:

{
  "prediction_label": "string",
  "confidence_level": "low | medium | high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}

Rules:
1. Base your explanation ONLY on the patient features, predicted class, and predicted probability provided.
2. Do NOT assume any information that is not present.
3. Do NOT invent additional risk factors.
4. Return ONLY valid JSON.
5. Do NOT add extra fields.
6. Do NOT include markdown or code fences.
"""

    user_prompt = f"""
Patient Features:
{json.dumps(result["features"], indent=2)}

Predicted Class: {prediction}

Predicted Probability: {result["predicted_probability"]:.4f}

Use the probability to decide whether the confidence level should be "low", "medium", or "high".

Return only valid JSON.
"""
# PII Guardrail
    guardrail_result = "Pass"

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        llm_response = None
        continue

# Call LLM

    llm_response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print("Raw LLM Response:")
    print(llm_response)

    try:

        parsed = json.loads(llm_response.strip())

        validate(
            instance=parsed,
            schema=schema
        )

        status = "Valid"

    except json.JSONDecodeError as e:

        parsed = {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": None
        }

        status = "Invalid JSON"
        print(e)

    except ValidationError as e:

        parsed = {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": None
        }

        status = "Schema Validation Failed"
        print(e)

    print("=" * 70)
    print("Features")
    print(result["features"])

    print("\nGuardrail Result:", guardrail_result)

    print("\nPredicted Class:", prediction)
    print("Probability:", round(result["predicted_probability"], 4))

    print("\nLLM Explanation")
    print(parsed)

    print("\nValidation Status:", status)

Raw LLM Response:
{
  "prediction_label": "No Stroke",
  "confidence_level": "low",
  "top_reason": "The patient has a predicted probability of 0.1312 for having a stroke.",
  "second_reason": "The patient is 65 years old and has hypertension and heart disease, but these factors are not sufficient to predict a stroke with high confidence.",
  "next_step": "Monitor the patient's health and consider regular check-ups for stroke risk assessment."
}
Features
{'age': 65, 'hypertension': 1, 'heart_disease': 1, 'bmi': 32.5, 'gender_Male': 1, 'gender_Other': 0, 'ever_married_Yes': 1, 'work_type_Never_worked': 0, 'work_type_Private': 1, 'work_type_Self-employed': 0, 'work_type_children': 0, 'Residence_type_Urban': 1, 'smoking_status_formerly smoked': 1, 'smoking_status_never smoked': 0, 'smoking_status_smokes': 0}

Guardrail Result: Pass

Predicted Class: No Stroke
Probability: 0.1312

LLM Explanation
{'prediction_label': 'No Stroke', 'confidence_level': 'low', 'top_reason': 'The patient has a 

In [ ]:
#Task - 4:
#Guardrails

#Import Regex
import re

#Create the PII Detection Function
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or    #True → Email or phone number found.
                                            #False → No PII found.
        re.search(phone_pattern, text)
    )

#Test the Guardrail(should be locked)

user_input = """
Patient Name: John
Email: john@gmail.com
Age: 65
"""

if has_pii(user_input):
    print("Input blocked: PII detected.")
else:
    print("No PII detected.")

#Test without PII:
user_input = """
Age: 65
Hypertension: Yes
BMI: 31.2
"""

if has_pii(user_input):
    print("Input blocked: PII detected.")
else:
    print("No PII detected. Proceed to LLM.")

Input blocked: PII detected.
No PII detected. Proceed to LLM.


**Task – 5:**
**End-to-End Pipeline Demonstration:**

The complete Track C pipeline was executed on three hand-crafted patient records. For each input, the trained machine learning model generated a prediction and probability. Before calling the LLM, a PII guardrail checked the input for email addresses and phone numbers. Since no PII was detected in the three patient records, all requests were allowed to proceed. The LLM returned structured JSON explanations, and each response was successfully validated against the predefined JSON schema.

| Input                                               | LLM Output                | Valid JSON | Pass/Block |
| --------------------------------------------------- | ------------------------- | ---------- | ---------- |
| Patient 1 (Age 65, Hypertension=1, Heart Disease=1) | JSON explanation returned | Pass       | Pass       |
| Patient 2 (Age 30, Healthy profile)                 | JSON explanation returned | Pass       | Pass       |
| Patient 3 (Age 78, Multiple risk factors)           | JSON explanation returned | Pass       | Pass       |

Demonstrate the guardrail blocking a request, include a separate example in the notebook where an input contains an email address and the result is:

| Input                         | LLM Output      | Valid JSON | Pass/Block |
| ----------------------------- | --------------- | ---------- | ---------- |
| Patient with `john@gmail.com` | Not sent to LLM | N/A        | **Block**  |


However, for the required three Track C feature vectors, they normally contain no PII, so all three would show Pass in the guardrail column.
